# Patch graph 到 CSG 的运行流程

预处理实现在 `csg/graph_construct.py`，primitive 候选、Boolean 搜索、验证和 OpenSCAD 生成实现在 `csg/csg_reconstruction.py`。本 notebook 仅负责配置与运行。

In [ ]:
from pathlib import Path
from pprint import pprint

from csg.graph_construct import plot_patch_graph, prepare_patch_graph
from csg.csg_reconstruction import (
    classify_boolean_operations,
    csg_to_openscad,
    generate_primitive_candidates,
    reconstruct_csg_tree,
    validate_csg,
)

In [ ]:
ply_path = Path(
    # 'example_data/final/ply/00023435_385b221a0d58490b84f3edee_trimesh_000/gt.ply'
    # 'example_data/final/ply/00023471_7f45ff9e8c754def8bb4b1cb_trimesh_000/gt.ply'
    # 'example_data/final/ply/00023792_30c31f050d2c40139e9b36ca_trimesh_001/gt.ply'
    # 'example_data/final/ply/00024036_de448e53313b4faa9dff2245_trimesh_003/gt.ply'
    # 'example_data/final/ply/00025183_021c990acfc648618a49bd47_trimesh_000/gt.ply' # nearly: free surface not all consistent
    # 'example_data/final/ply/00025611_37f3a717a9b842cfbe5f6005_trimesh_002/gt.ply'
    # 'example_data/final/ply/00025710_6fe81cf35e2740b3bbde9aa6_trimesh_020/gt.ply'
    'example_data/final/ply/00025728_6fe81cf35e2740b3bbde9aa6_trimesh_038/gt.ply'
    # 'example_data/final/ply/00140553_e2c0841b6c86e3bfdcc8c477_trimesh_000/gt.ply'
    # 'example_data/final/ply/00140750_0da4d13f288d4cd70deecf20_trimesh_001/gt.ply'
    # 'example_data/final/ply/00140919_3023c3cfd7d6b1a80700e17e_trimesh_000/gt.ply'
    # 'example_data/final/ply/00140936_9a2b76f7e489ea90f5e41ef8_trimesh_007/gt.ply'
    
    # not consistent
    # 'example_data/final/ply/00023576_35c7024f831f44048104c331_trimesh_000/gt.ply'
    # 'example_data/final/ply/00024040_de448e53313b4faa9dff2245_trimesh_007/gt.ply'
    # 'example_data/final/ply/00024082_de448e53313b4faa9dff2245_trimesh_049/gt.ply'
    # 'example_data/final/ply/00024132_a087bf84d26046349bbbd6f6_trimesh_000/gt.ply'    
    # 'example_data/final/ply/00024961_e26f4b039c6c48c9878b21fe_trimesh_000/gt.ply'
    # 'example_data/final/ply/00025633_2aaf78a0f41a47b1bae8c447_trimesh_003/gt.ply'
    # 'example_data/final/ply/00025749_c01a880ceccb4e05908f8255_trimesh_000/gt.ply'
    # 'example_data/final/ply/00026133_8010d7708c06499e9e99ce1c_trimesh_008/gt.ply'
    
    # fail
    # 'example_data/final/ply/00024792_34a17822747a4b20a8c2954b_trimesh_009/gt.ply'
    
    # chamfer & fillet
    # 'example_data/final/ply/00023582_9c917172a61b472fb0e6ae3c_trimesh_002/gt.ply' # lack chamfer & fillet
    # 'example_data/final/ply/00025063_9347be1a9dad4e70852dcce2_trimesh_000/gt.ply' # lack chamfer
    # 'example_data/final/ply/00140710_33073a6cb55fadd20ec0e3fb_trimesh_001/gt.ply' # lack chamfer & fillet
    # 'example_data/final/ply/00140400_e4a5e7862a9beda824b2e00c_trimesh_003/gt.ply' # lack chamfer
    # 'example_data/final/ply/00140756_eaf6606df2a105f356a99677_trimesh_000/gt.ply' # fillet affect modeling
    # 'example_data/final/ply/00140982_f2e481e34cff3a450799acac_trimesh_000/gt.ply' # fillet fail; lack hole; wrong segment
    # 'example_data/final/ply/00141097_57222034e4b08229a097b208_trimesh_010/gt.ply' # lack chamfer & fillet; single thin-walled after diff
    
    # free surface/body
    # 'example_data/final/ply/00026055_11394f3abb83419b88f6eed9_trimesh_000/gt.ply'
    # 'example_data/final/ply/00140923_0e818d7ff8780682567c824e_trimesh_000/gt.ply'
    # 'example_data/final/ply/00140967_631426bb5199fa39fc8eeb1c_trimesh_008/gt.ply'
    
    # scaling
    # 'example_data/final/ply/00024448_274a7c9d6def4a699961a747_trimesh_002/gt.ply'
    
)

min_patch_faces = 1
include_torus = True
run_voxel_validation = True
print_patches = True
visualize_patches = True
export_openscad = False
openscad_path = Path("example_results") / f"{ply_path.parent.name}.scad"

## 1. 分割读取、primitive 拟合与 patch graph

In [ ]:
target_mesh, patches, patch_results, patch_graph = prepare_patch_graph(
    ply_path,
    min_faces=min_patch_faces,
)

if print_patches:
    print(f"patches: {len(patches)}")
    print(f"shared boundary edges: {len(patch_graph['edges'])}")
    for result in patch_results:
        print(
            f"patch {result['patch_index']:>3}: "
            f"{result['type']:<10} fit={result['fit_rate']:.4f} "
            f"faces={len(result['face_id'])} "
            f"neighbors={result['neighboring_patches']}"
        )

if visualize_patches:
    plot_patch_graph(target_mesh, patch_results)

## 2. Primitive candidates、Boolean 分类与 CSG 搜索

In [ ]:
primitive_candidates = generate_primitive_candidates(
    patch_results,
    patch_graph,
    include_torus=include_torus,
)
classify_boolean_operations(primitive_candidates, patch_results)

csg_ir, selected_primitives, search_report = reconstruct_csg_tree(
    primitive_candidates,
    patch_results,
)
validation_report = validate_csg(
    csg_ir,
    selected_primitives,
    patch_results,
    target_mesh=target_mesh if run_voxel_validation else None,
)
openscad_source = csg_to_openscad(csg_ir)

## 3. 结果

In [ ]:
candidate_counts = {
    primitive_type: sum(
        candidate.primitive_type == primitive_type
        for candidate in primitive_candidates
    )
    for primitive_type in ("CUBE", "CYLINDER", "CONE", "SPHERE", "TORUS")
}

print("candidate counts:")
pprint(candidate_counts)
print("\nselected primitives:")
for candidate in selected_primitives:
    print(
        candidate.operation,
        candidate.primitive_type,
        candidate.patch_ids,
        f"error={candidate.fitting_error:.6f}",
        f"confidence={candidate.confidence:.4f}",
    )
print("\nsearch report:")
pprint(search_report)
print("\nvalidation report:")
pprint(validation_report)
print("\nCSG IR:")
pprint(csg_ir)
print("\nOpenSCAD:")
print(openscad_source)

if export_openscad:
    openscad_path.parent.mkdir(parents=True, exist_ok=True)
    openscad_path.write_text(openscad_source + "\n", encoding="utf-8")
    print(f"saved: {openscad_path}")